# Raw106 NMR score derivation


In [ ]:
from pathlib import Path
for required_source in ("score106.py", "metrics.py"):
    required_path = Path.cwd().parent / "sources" / required_source
    if not required_path.is_file():
        raise FileNotFoundError(f"Required analysis module not found: {required_path}")

from pathlib import Path
import os, json, copy, hashlib, shutil, ast
PACKAGE = Path.cwd().parent

PROJECT = Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
BASE_RUN = os.environ.get('CHD_SCORE106_BASE_RUN', 'final_rerun_2026')
NEW_RUN = os.environ.get('CHD_SCORE106_RUN', 'reviewer_raw106_score')
COHORT = os.environ.get('CHD_SCORE106_COHORT', 'prs_raw106_complete')

BOOTSTRAPS = 1000
os.environ.update(CHD_OUTPUT_ROOT=str(PROJECT), CHD_RUN_ID=BASE_RUN, CHD_COHORT_ID=COHORT,
                  CHD_SENSITIVITY_OUTPUT_DIR=str(PROJECT/'runs'/BASE_RUN),
                  CHD_SHARED_NOTEBOOK=str((Path.cwd() / '00_shared_config.ipynb')), CHD_BOOTSTRAP_N=str(BOOTSTRAPS))
get_ipython().run_line_magic('run', '"'+str((Path.cwd() / '00_shared_config.ipynb'))+'"')
df_train,df_val,df_test,final_feature_sets,base_manifest=validate_pipeline_artifacts(True)
BASE = PROJECT/'runs'/BASE_RUN
frames={'train':df_train,'val':df_val,'test':df_test}
print('Verified existing partitions:', {k:len(v) for k,v in frames.items()})


In [ ]:
if NEW_RUN == BASE_RUN:
    raise ValueError("The Score106 run must differ from its reference run.")
SCORE106_SOURCE = PACKAGE / "sources" / "score106.py"
if not SCORE106_SOURCE.is_file():
    raise FileNotFoundError(f"Required analysis module not found: {SCORE106_SOURCE}")

DEST=PROJECT/'runs'/NEW_RUN
if DEST.exists():
    raise FileExistsError(f'{DEST} already exists. Existing runs are protected from overwrite.')
DEST.mkdir(parents=True)
shutil.copytree(BASE/'01_cohort',DEST/'01_cohort')
os.environ.update(CHD_RUN_ID=NEW_RUN, CHD_SENSITIVITY_OUTPUT_DIR=str(DEST))
get_ipython().run_line_magic('run','"'+str((Path.cwd() / '00_shared_config.ipynb'))+'"')
nmr106_cols=list(base_manifest['nmr106_columns'])
assert len(nmr106_cols)==106 and len(set(nmr106_cols))==106
score_frames={k:v.copy() for k,v in frames.items()}

if COHORT=='coverage90':
    raw_parts={}
    wanted=set(nmr106_cols)
    for entry in base_manifest['input_files']:
        path=Path(entry['path'])
        if sha256_file(path)!=entry['sha256']: raise RuntimeError(f'Input file digest changed: {path}')
        headers=pd.read_csv(path,nrows=0).columns.tolist()
        eid=find_eid_col(headers)
        mapping={c:str(parse_ukb_col(c)[0]) for c in headers if parse_ukb_col(c) is not None
                 and parse_ukb_col(c)[1:]==(0,0) and str(parse_ukb_col(c)[0]) in wanted}
        if not mapping: continue
        raw=pd.read_csv(path,usecols=[eid]+list(mapping),low_memory=False).rename(columns={eid:'Eid',**mapping})
        if raw.Eid.duplicated().any(): raise RuntimeError('Duplicate raw EIDs')
        raw=raw.set_index('Eid')
        for c in mapping.values():
            s=pd.to_numeric(raw[c],errors='coerce')
            if c in raw_parts and not raw_parts[c].equals(s): raise RuntimeError('Conflicting raw columns: '+c)
            raw_parts[c]=s
    if set(raw_parts)!=wanted: raise RuntimeError('Missing raw106 columns for nested imputation')
    for k,f in score_frames.items():
        for c,s in raw_parts.items(): f[c]=f.Eid.map(s)
exec(compile(SCORE106_SOURCE.read_text(encoding="utf-8"), 'score106.py','exec'), globals())
values=fit_nmr_score_nested(score_frames['train'],score_frames['val'],score_frames['test'])
train_score,val_score,test_score,cv,coef,prep,metadata,boundary=values
metadata.update(score_name='NMR_score106', predictor_contract_name='Raw106', predictor_field_count=106,
                source_definition='Prespecified Raw106 panel', source_definition_version='reviewer_same_space_106_v1',
                downstream_column_alias='NMR_score', reference_run=BASE_RUN)
atomic_write_csv(cv,Path(AUDIT_DIR)/'nmr_score106_inner_cv_grid.csv',index=False)
atomic_write_csv(coef,Path(SCORE_ARTIFACT_DIR)/'nmr_score106_coefficients.csv',index=False)
atomic_write_csv(prep,Path(PREPROCESSING_ARTIFACT_DIR)/'nmr_score106_train_preprocessing.csv',index=False)
atomic_write_json(metadata,Path(SCORE_ARTIFACT_DIR)/'nmr_score_metadata.json')
manifest=copy.deepcopy(base_manifest)
manifest.update(run_id=NEW_RUN,nmr_score=metadata,reference_run=BASE_RUN,
                revision_role='same_Raw106_candidate_space_score_comparison')
score_registry_path=Path(SCORE_ARTIFACT_DIR)/'score106_active_registry.csv'
atomic_write_csv(pd.DataFrame([{'Score':'NMR_score106','Candidate_N':106,'Selected_N':metadata['selected_n'],
                               'Downstream_Alias':'NMR_score','Fit_Split':'train'}]),score_registry_path,index=False)
manifest['score_registry']={'csv':str(score_registry_path),'csv_sha256':sha256_file(score_registry_path)}
for k,score_values in zip(['train','val','test'],[train_score,val_score,test_score]):
    f=frames[k].copy(); f['NMR_score']=score_values
    assert eid_digest(f)==base_manifest['splits'][k]['eid_sha256']
    assert dataframe_digest(f,['Eid','Time','Event'])==base_manifest['splits'][k]['outcome_sha256']
    path=split_artifact_path(k); atomic_write_csv(f,path,index=False)
    saved = pd.read_csv(path, low_memory=False)
    if len(saved) != len(f) or eid_digest(saved) != eid_digest(f):
        raise RuntimeError(f"{k}: EID order or row count changed during serialization")
    if not np.array_equal(saved.Event.to_numpy(), f.Event.to_numpy()):
        raise RuntimeError(f"{k}: Event changed during serialization")
    if not np.isfinite(saved.Time).all() or not np.allclose(saved.Time, f.Time, rtol=0, atol=1e-12):
        raise RuntimeError(f"{k}: Time changed during serialization")
    manifest["splits"][k].update(
        file_sha256=sha256_file(path),
        outcome_sha256=dataframe_digest(saved, ["Eid", "Time", "Event"]),
    )
atomic_write_json(manifest,PIPELINE_MANIFEST_PATH)
# Historical copied auxiliary registries are reference artifacts, not active score definitions.
atomic_write_json({'active_score':metadata,'historical_auxiliary_registries_from':str(BASE),
                  'authoritative_manifest':str(PIPELINE_MANIFEST_PATH)},DEST/'SCORE106_PROVENANCE.json')
df_train,df_val,df_test,final_feature_sets,pipeline_manifest=validate_pipeline_artifacts(True)
print('Score106 ready; exact EID order, outcomes, non-score features preserved:',DEST)


In [ ]:
from pathlib import Path
import os, json
PACKAGE=Path.cwd().parent
PROJECT=Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
NEW_RUN=os.environ.get('CHD_SCORE106_RUN', 'reviewer_raw106_score')
COHORT=os.environ.get('CHD_SCORE106_COHORT', 'prs_raw106_complete')
os.environ.update(CHD_OUTPUT_ROOT=str(PROJECT),CHD_RUN_ID=NEW_RUN,CHD_COHORT_ID=COHORT,
                  CHD_SENSITIVITY_OUTPUT_DIR=str(PROJECT/'runs'/NEW_RUN),
                  CHD_SHARED_NOTEBOOK=str((Path.cwd() / '00_shared_config.ipynb')))
for stage in ['02_incremental_value.ipynb', '03_nmr_representation_comparison.ipynb', '04_survival_model_comparison.ipynb', '05_calibration_dca_and_joint_risk.ipynb']:
    print('START',stage,flush=True)
    get_ipython().run_line_magic('run','"'+str(Path.cwd() / stage)+'"')
    print('DONE',stage,flush=True)


In [ ]:
def stamp(message):
    print(message, flush=True)

def atomic_json(path, payload):
    atomic_write_json(payload, path)

get_ipython().run_line_magic("run", f'"{os.environ["CHD_SHARED_NOTEBOOK"]}"')
df_train, df_val, df_test, final_feature_sets, pipeline_manifest = validate_pipeline_artifacts(True)

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
from lifelines.statistics import proportional_hazard_test

INTERACTION_ROOT = Path(RESULTS_OUTPUT_DIR) / "supplementary_prs_nmr_score_interaction"
INTERACTION_ROOT.mkdir(parents=True, exist_ok=True)

interaction_features = list(dict.fromkeys(list(final_feature_sets["Clinical_only"]) + [PRS_COL, "NMR_score"]))
X_train_i, X_val_i, X_test_i, interaction_actual = prepare_X_train_val_test(
    df_train, df_val, df_test, interaction_features, scale=True
)
if PRS_COL not in interaction_actual or "NMR_score" not in interaction_actual:
    raise RuntimeError("PRS or NMR_score missing from continuous interaction design")

split_payloads = {
    "train": (df_train, X_train_i),
    "validation": (df_val, X_val_i),
    "test": (df_test, X_test_i),
}
pipeline_sha = current_pipeline_sha256()

def _interaction_signature(split_name, frame, columns):
    payload = {
        "analysis": "continuous_PRS_x_NMR_score",
        "split": split_name,
        "pipeline_manifest_sha256": pipeline_sha,
        "eid_sha256": eid_digest(frame),
        "outcome_sha256": dataframe_digest(frame, ["Eid", "Time", "Event"]),
        "columns": list(columns) + ["PRS_x_NMR_score"],
        "preprocessing": "training-only prepare_X_train_val_test(scale=True)",
        "model": "unpenalized Cox PH; robust sandwich SE; Efron ties; batch_mode",
    }
    signature = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    return payload, signature

def _fit_interaction_split(split_name):
    frame, design = split_payloads[split_name]
    design = design.reset_index(drop=True).copy()
    design["PRS_x_NMR_score"] = design[PRS_COL].to_numpy(float) * design["NMR_score"].to_numpy(float)
    fit_frame = pd.concat([frame[["Time","Event"]].reset_index(drop=True), design], axis=1)
    payload, signature = _interaction_signature(split_name, frame, design.columns[:-1])
    csv_path = INTERACTION_ROOT / f"continuous_prs_x_nmr_score_{split_name}.csv"
    json_path = INTERACTION_ROOT / f"continuous_prs_x_nmr_score_{split_name}.json"
    if csv_path.exists() and json_path.exists():
        saved = json.loads(json_path.read_text(encoding="utf-8"))
        if saved.get("signature") == signature:
            stamp(f"RESUME interaction | split={split_name} | signed cache valid")
            return pd.read_csv(csv_path).iloc[0].to_dict()

    started = time.time(); stamp(f"START interaction Cox | split={split_name} | N={len(frame):,} | events={int(frame.Event.sum()):,} | features={design.shape[1]}")
    model = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(fit_frame, duration_col="Time", event_col="Event", robust=True, batch_mode=True, show_progress=False)
    term = model.summary.loc["PRS_x_NMR_score"]

    # PH diagnostic is required only for the locked-test association model.
    if split_name == "test":
        ph = proportional_hazard_test(model, fit_frame, time_transform="rank").summary
        ph_p = float(ph.loc["PRS_x_NMR_score", "p"])
        ph_n = int(ph["p"].notna().sum())
        ph_bonferroni = min(1.0, ph_p * ph_n)
    else:
        ph_p, ph_n, ph_bonferroni = np.nan, 0, np.nan
    row = {
        "Split": split_name, "N": int(len(frame)), "Event_N": int(frame.Event.sum()),
        "Term": "PRS_x_NMR_score", "Beta": float(term["coef"]), "SE": float(term["se(coef)"]),
        "HR": float(term["exp(coef)"]), "HR_CI_Lower": float(term["exp(coef) lower 95%"]),
        "HR_CI_Upper": float(term["exp(coef) upper 95%"]), "P_Value": float(term["p"]),
        "PH_P_Raw": ph_p, "PH_Test_N": ph_n, "PH_P_Bonferroni": ph_bonferroni,
        "Scale": "interaction between training-standardized PRS and training-standardized NMR_score",
        "Adjustment": "all Clinical_only columns plus PRS and NMR_score main effects",
        "Pipeline_Manifest_SHA256": pipeline_sha, "Signature": signature,
        "Elapsed_Minutes": (time.time()-started)/60,
    }
    pd.DataFrame([row]).to_csv(csv_path, index=False)
    atomic_json(json_path, {"signature": signature, "signature_payload": payload})
    stamp(f"DONE interaction Cox | split={split_name} | HR={row['HR']:.5f} | P={row['P_Value']:.4g} | elapsed={row['Elapsed_Minutes']:.1f} min")
    return row

interaction_splits = ["train", "validation", "test"]
interaction_rows = Parallel(
    n_jobs=min(3, CPU_WORKERS), backend="loky", verbose=10,
    pre_dispatch=min(3, CPU_WORKERS), batch_size=1, max_nbytes="50M",
)(delayed(_fit_interaction_split)(split_name) for split_name in interaction_splits)

interaction_results = pd.DataFrame(interaction_rows)
interaction_results.to_csv(INTERACTION_ROOT / "continuous_prs_x_nmr_score_all_splits.csv", index=False)
display(interaction_results[["Split","N","Event_N","HR","HR_CI_Lower","HR_CI_Upper","P_Value","PH_P_Bonferroni","Elapsed_Minutes"]])
stamp(f"INTERACTION COMPLETE | {INTERACTION_ROOT}")
